# Stateful signal pipeline

Самостоятельный production-shaped pipeline без зависимости от research notebook и его артефактов:

`новые данные → update_models_if_due → get_signal → filter_signal → final backtest`.

Исторический прогон ниже буквально воспроизводит ежедневную production-последовательность. Rule-движки переоптимизируют только concrete thresholds внутри уже выбранной архитектуры и cadence. ML-движки используют фиксированный `HistGradientBoosting`, ежегодное переобучение и калибровку probability threshold на прошлом validation-окне. Будущие targets доступны только после созревания горизонта.

In [ ]:
from datetime import date
from pathlib import Path
import json
import sys

import pandas as pd

candidate_roots = [Path.cwd(), Path.cwd() / 'prod', Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidate_roots if (p / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Не найден корень репозитория с папкой src')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cbr_loader import CURRENCIES, load_cbr_history
from src.features import build_features
from src.market_data import build_daily_market_panel
from src.meta_model import (
    build_meta_candidates, fit_logistic_meta_model, logistic_meta_model,
    run_meta_model,
)
from src.outcomes import add_future_outcomes
from src.production_config import (
    FIXED_INDICATORS, INDICATOR_SPACES, RULE_MIN_SIGNALS_PER_WEEK,
    ML_CONFIG, ML_FEATURE_NAMES, ML_MIN_SIGNALS_PER_WEEK,
    ML_MODEL_TYPE, ML_RETRAIN_MONTHS, ML_VALIDATION_MONTHS,
    PRODUCTION_CURRENCIES,
    PRODUCTION_HORIZONS, PRODUCTION_TARGET_FAMILIES, TRAIN_WINDOW_MONTHS,
    fixed_indicator_registry,
)
from src.production_pipeline import (
    engine_state_registry, filter_signal, get_signal, initialize_engine_states,
    replay_engine_signals, save_engine_states,
)
from src.signal_backtest import backtest_signal_stream, build_evaluation_universe
from src.visualization import plot_benefit_distribution, plot_oos_uplift, plot_signal_series
from src.targets import build_targets

START_DATE = date(2020, 1, 1)
END_DATE = date.today()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'cbr'

# Base OOS начинается в 2022. Meta train заканчивается до 2024,
# а финальный backtest всей связки начинается в 2025.
BASE_OOS_START = pd.Timestamp('2022-01-01')
META_TRAIN_END = pd.Timestamp('2024-01-01')
FINAL_BACKTEST_START = pd.Timestamp('2025-01-01')
META_VALIDATION_MONTHS = 12
META_MIN_SIGNALS_PER_WEEK = 1.0
META_MAX_SIGNALS_PER_WEEK = 2.0

## 1. Данные, causal features и targets

Features строки используют только сведения, доступные к её `available_at`. Future outcomes и targets нужны для исторического обучения и оценки; они не входят в JSON сигнала.

In [ ]:
cbr_history = load_cbr_history(
    start_date=START_DATE,
    end_date=END_DATE,
    currencies=CURRENCIES,
    raw_dir=RAW_DIR,
)
rates = (
    cbr_history.pivot(index='available_at', columns='currency', values='normalized_rate')
    .reindex(columns=list(CURRENCIES))
    .sort_index()
)
rates.columns.name = None
market_panel = build_daily_market_panel(rates)
features = build_features(market_panel)
outcomes = add_future_outcomes(features, horizons=PRODUCTION_HORIZONS)
dataset, target_registry = build_targets(outcomes, horizons=PRODUCTION_HORIZONS)
scoring_data = dataset.loc[
    dataset['is_update_day'] & dataset['currency'].isin(PRODUCTION_CURRENCIES)
].copy()

pd.DataFrame({
    'first_available_at': [scoring_data['available_at'].min()],
    'last_available_at': [scoring_data['available_at'].max()],
    'rows': [len(scoring_data)],
})

## 2. Реестр движков

Создаётся по одному rule- и одному ML-движку для каждой `currency × target_family × horizon`. Пустое состояние знает, когда впервые обучиться; после обучения в нём находятся конкретное правило либо веса модели, версия, даты train и следующий срок обновления.

In [ ]:
production_rule_registry = fixed_indicator_registry()
states = initialize_engine_states(
    rule_configurations=FIXED_INDICATORS,
    target_registry=target_registry,
    currencies=PRODUCTION_CURRENCIES,
    target_families=PRODUCTION_TARGET_FAMILIES,
    first_score_date=BASE_OOS_START,
    train_months=TRAIN_WINDOW_MONTHS,
    ml_feature_names=ML_FEATURE_NAMES,
    ml_model_type=ML_MODEL_TYPE,
    ml_retrain_months=ML_RETRAIN_MONTHS,
)

assert len(states) == 2 * len(PRODUCTION_CURRENCIES) * len(PRODUCTION_TARGET_FAMILIES) * len(PRODUCTION_HORIZONS)
production_rule_registry

## 3. Ежедневный исторический replay базовых движков

Для каждой даты выполняется один и тот же порядок:

1. `update_models_if_due` переобучает только просроченные движки и только на уже созревших targets из прошлого.
2. `get_signal` возвращает JSON-совместимый score всех движков, включая `signal=false` и технический статус.
3. Результатом является только неизменяемый `raw_signal_stream`; метамодель запускается отдельным этапом ниже.

Никакие итоговые метрики внутри этого цикла не оптимизируются.

In [ ]:
replay = replay_engine_signals(
    scoring_data,
    states=states,
    first_score_date=BASE_OOS_START,
    indicator_spaces=INDICATOR_SPACES,
    rule_min_signals_per_week=RULE_MIN_SIGNALS_PER_WEEK,
    ml_validation_months=ML_VALIDATION_MONTHS,
    ml_min_signals_per_week=ML_MIN_SIGNALS_PER_WEEK,
    ml_model_type=ML_MODEL_TYPE,
    ml_model_config=ML_CONFIG,
)

## 4. Техническое состояние после replay

Это не таблица качества. Она отвечает только на эксплуатационные вопросы: какой артефакт сейчас активен, на каких зрелых данных обучен и когда должен обновиться. `training_audit` позволяет проверить, что обновления действительно происходили по заданному cadence.

In [ ]:
current_engine_states = engine_state_registry(replay.states)
current_engine_states

In [ ]:
replay.training_audit[[
    'as_of', 'engine_id', 'fitted', 'trained_through', 'next_retrain_at'
]].tail(20)

## 5. Логистическая метамодель

Метамодель получает только raw OOS-предикты базовых движков. Она обучается на OOS-кандидатах 2022–2023, использует 2023 как внутреннее validation-окно для выбора единого probability threshold, затем применяется к финальному потоку с `2025-01-01`. Базовые сигналы при этом не переобучаются и не изменяются.

In [ ]:
raw_signal_stream = replay.raw_signals.copy()
raw_signal_stream['available_at'] = pd.to_datetime(raw_signal_stream['available_at'])

all_evaluation_universe = build_evaluation_universe(
    scoring_data,
    target_registry=target_registry,
    target_families=PRODUCTION_TARGET_FAMILIES,
    currencies=PRODUCTION_CURRENCIES,
    start_date=BASE_OOS_START,
)
meta_candidates = build_meta_candidates(raw_signal_stream)
meta_keys = ['available_at', 'currency', 'scenario', 'target_family', 'target', 'horizon']
meta_training_data = meta_candidates.merge(
    all_evaluation_universe[meta_keys + ['target_value']],
    on=meta_keys, how='left', validate='many_to_one',
).dropna(subset=['target_value'])

fitted_meta_model = fit_logistic_meta_model(
    meta_training_data,
    train_end=META_TRAIN_END,
    validation_months=META_VALIDATION_MONTHS,
    min_signals_per_week=META_MIN_SIGNALS_PER_WEEK,
    max_signals_per_week=META_MAX_SIGNALS_PER_WEEK,
)

final_raw_signal_stream = raw_signal_stream.loc[
    raw_signal_stream['available_at'].ge(FINAL_BACKTEST_START)
].copy()
final_signal_stream = run_meta_model(
    final_raw_signal_stream,
    meta_model=logistic_meta_model,
    config=fitted_meta_model,
)

# Отдельный production-style вызов для последней даты: raw JSON → filter JSON.
latest_date = raw_signal_stream['available_at'].max()
latest_raw_vector = raw_signal_stream.loc[
    raw_signal_stream['available_at'].eq(latest_date)
].to_dict(orient='records')
latest_filtered_events = filter_signal(
    latest_raw_vector,
    meta_model=logistic_meta_model,
    meta_config=fitted_meta_model,
)
print(f'Raw engine scores: {len(latest_raw_vector)}')
print(f'Filtered events: {len(latest_filtered_events)}')

meta_model_summary = pd.DataFrame([{
    'trained_at': fitted_meta_model.trained_at,
    'trained_through': fitted_meta_model.trained_through,
    'validation_start': fitted_meta_model.validation_start,
    'validation_end': fitted_meta_model.validation_end,
    'threshold': fitted_meta_model.threshold,
    'validation_precision': fitted_meta_model.validation_precision,
    'validation_lift': fitted_meta_model.validation_lift,
    'validation_signals_per_week': fitted_meta_model.validation_signals_per_week,
}])
meta_model_summary

## 6. Единственная итоговая оценка — после метамодели

`signal_backtest_summary` считает результаты конечных событий логистической метамодели начиная с `FINAL_BACKTEST_START = 2025-01-01`. В отчёте ровно 65 строк: 5 по валютам, 25 по валютам и горизонтам, 50 по валютам, горизонтам и target family. Промежуточные rule/ML scores и validation-метрики не являются финальным backtest.

In [ ]:
evaluation_universe = build_evaluation_universe(
    scoring_data,
    target_registry=target_registry,
    target_families=PRODUCTION_TARGET_FAMILIES,
    currencies=PRODUCTION_CURRENCIES,
    start_date=FINAL_BACKTEST_START,
)
signal_backtest_summary, signal_backtest_rows = backtest_signal_stream(
    final_signal_stream,
    evaluation_universe=evaluation_universe,
)
signal_backtest_summary

## 7. Опциональное сохранение production state

На реальном сервисе состояние сохраняется после успешного обновления и загружается при старте процесса. В notebook запись намеренно выключена: раскомментируйте только если нужен локальный snapshot.

In [ ]:
# STATE_PATH = PROJECT_ROOT / 'models' / 'engine_states.joblib'
# save_engine_states(replay.states, STATE_PATH)

# 8. Визуализации

In [ ]:
# Отдельная секция графика конечных сигналов после метамодели.
import matplotlib.pyplot as plt
PRESENTATION_CURRENCY = 'AMD'
plot_signal_series(
    scoring_data,
    final_signal_stream,
    currency=PRESENTATION_CURRENCY,
    start_date="2025-01-01",
)
plt.tight_layout()
plt.show()

plot_oos_uplift(
    signal_backtest_summary.loc[signal_backtest_summary['scope'].eq('currency+horizon+target_family')],
    uplift_column='lift',
    title='Финальный OOS uplift после meta-модели',
)
plt.tight_layout()
plt.show()

signal_benefit = signal_backtest_rows.loc[signal_backtest_rows['signal']].copy()
if 'benefit_bps' in signal_benefit.columns:
    plot_benefit_distribution(signal_benefit)
    plt.tight_layout()
    plt.show()